<a href="https://colab.research.google.com/github/saranoor/chatGPT/blob/master/policy_2_create_qa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!pip install openai

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.1/70.1 KB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.2/199.2 KB 22.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.8/158.8 KB 16.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.2/114.2 KB 8.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.6/264.6 KB 26.1 MB/s eta 0:00:00


In [7]:
import openai

In [8]:
API_KEY = '' # Add your API key here
openai.api_key = API_KEY

In [9]:
import pandas as pd
df = pd.read_excel('/content/10pearls_sections_text_embeddings.xlsx')
df['context'] = df.title + "\n" + df.heading + "\n\n" + df.content
df.head()

,Unnamed: 0,title,heading,content,combined,n_tokens,embeddings,context
0,0,10Pearls Policies and Procedures,OUR MISSION,TenPearls (Private) Limited is founded on the ...,Title: 10Pearls Policies and Procedures; Headi...,106,"[-0.0033130680676549673, -0.02280309423804283,...",10Pearls Policies and Procedures\nOUR MISSION\...
1,1,Definitions in 10Pearls Policies and Procedures,DEFINITIONS,In this Manual the following words and express...,Title: Definitions in 10Pearls Policies and Pr...,39,"[-0.021558966487646103, -0.0056012338027358055...",Definitions in 10Pearls Policies and Procedure...
2,2,Definitions in 10Pearls Policies and Procedures,Company,“Company” shall mean TenPearls (Private) Limited;,Title: Definitions in 10Pearls Policies and Pr...,28,"[-0.01174114178866148, -0.004361748229712248, ...",Definitions in 10Pearls Policies and Procedure...
3,3,Definitions in 10Pearls Policies and Procedures,PULSE,“PULSE” shall mean the Company online portal a...,Title: Definitions in 10Pearls Policies and Pr...,34,"[-0.033730316907167435, 0.0022980982903391123,...",Definitions in 10Pearls Policies and Procedure...
4,4,Definitions in 10Pearls Policies and Procedures,Employee,“Employee(s)” shall mean all employees of the ...,Title: Definitions in 10Pearls Policies and Pr...,51,"[-0.026462620124220848, 0.0019038093741983175,...",Definitions in 10Pearls Policies and Procedure...


### Create Qs based upon the context

In [10]:
import openai

def get_questions(context):
    try:
        response = openai.Completion.create(
            engine="davinci-instruct-beta-v3",
            prompt=f"Write questions based on the text below\n\nText: {context}\n\nQuestions:\n1.",
            temperature=0,
            max_tokens=257,
            top_p=1,
            frequency_penalty=0,
            presence_penalty=0,
            stop=["\n\n"]
        )
        return response['choices'][0]['text']
    except:
        return ""


df['questions']= df.context.apply(get_questions)
df['questions'] = "1." + df.questions
print(df[['questions']].values[0][0])

1. What is TenPearls' mission?
2. What are the principles that TenPearls is founded on?
3. What does TenPearls do to create value for their customers?
4. What is TenPearls' goal for their employees?
5. How does TenPearls give back to society?


In [12]:
df.to_excel('policy_embeddings_with_qs.xlsx')

### Create Answere based on the context

In [13]:
def get_answers(row):
    try:
        response = openai.Completion.create(
            engine="davinci-instruct-beta-v3",
            prompt=f"Write answer based on the text below\n\nText: {row.context}\n\nQuestions:\n{row.questions}\n\nAnswers:\n1.",
            temperature=0,
            max_tokens=257,
            top_p=1,
            frequency_penalty=0,
            presence_penalty=0
        )
        return response['choices'][0]['text']
    except Exception as e:
        print (e)
        return ""


df['answers']= df.apply(get_answers, axis=1)
df['answers'] = "1." + df.answers
df = df.dropna().reset_index().drop('index',axis=1)
print(df[['answers']].values[0][0])

1. TenPearls' mission is to generate opportunities, value and wealth which can change the world in a positive way.
2. TenPearls is founded on the principles of creating opportunities, value and wealth which can change the world in a positive way.
3. TenPearls creates value for their customers by helping them deliver on their technical and business challenges through Automation, Innovation and Integration.
4. TenPearls' goal for their employees is to groom them to be the next leaders, giving back to society and the world at large.
5. TenPearls gives back to society by donating money to charity, creating jobs and opportunities, and leaving a legacy of grooming employees to be the next leaders.


In [14]:
df.to_excel('policy_embeddings_with_qs_ans.xlsx')

### Search File

In [17]:
df.columns

Index(['Unnamed: 0', 'title', 'heading', 'content', 'combined', 'n_tokens',
       'embeddings', 'context', 'questions', 'answers'],
      dtype='object')

In [24]:
df = df[df.n_tokens<2000]
df[['context', 'n_tokens']].rename(columns={'context':'text','n_tokens':'metadata'}).to_json('policy_search.jsonl', orient='records', lines=True)

In [28]:
search_file = openai.File.create(
  file=open("policy_search.jsonl"),
  purpose='search'
)
policy_search_fileid = search_file['id']

InvalidRequestError: ignored